In [19]:
import pickle as pkl
import pandas as pd #v2.2.3
import numpy as np # v2.2.0
import os
from collections import Counter
import plotly.express as px #plotly v5.24.1
from operator import methodcaller

In [20]:
directory = "../data/MTBLS28_urinaryMetabolomic/MTBLS28_187294/"
fileSamples = "s_MTBLS28.txt"
filePOSdata = "m_MTBLS28_POS_v2_maf.tsv"
fileNEGdata = "m_MTBLS28_NEG_v2_maf.tsv"


### Metadata

In [21]:
df_metadata = pd.read_csv(directory+fileSamples, delimiter="\t")
df_metadata

,Source Name,Characteristics[Organism],Term Source REF,Term Accession Number,Characteristics[Organism part],Term Source REF.1,Term Accession Number.1,Characteristics[Variant],Term Source REF.2,Term Accession Number.2,...,Term Accession Number.4,Factor Value[Race],Term Source REF.5,Term Accession Number.5,Factor Value[Gender],Term Source REF.6,Term Accession Number.6,Factor Value[Sample Type],Term Source REF.7,Term Accession Number.7
0,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Male,NaN,NaN,Control,NaN,NaN
1,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Female,NaN,NaN,Case,NaN,NaN
2,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Male,NaN,NaN,Control,NaN,NaN
3,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Male,NaN,NaN,Control,NaN,NaN
4,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Female,NaN,NaN,Control,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2005,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,African American,NaN,NaN,Female,NaN,NaN,Case,NaN,NaN
2006,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Male,NaN,NaN,Control,NaN,NaN
2007,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Male,NaN,NaN,Case,NaN,NaN
2008,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Female,NaN,NaN,Control,NaN,NaN


In [22]:
df_metadata = df_metadata.rename(columns={'Sample Name': 'Sample_Name'})
# Remove lines based on the Sample_Name column content (remove if NEG is present)
df_metadata[~df_metadata.Sample_Name.str.contains("NEG")].to_csv(directory+"s_MTBLS28_POS.csv", sep=";")
# Remove lines based on the Sample_Name column content (remove if POS is present)
df_metadata[~df_metadata.Sample_Name.str.contains("POS")].to_csv(directory+"s_MTBLS28_NEG.csv", sep=";")


### Data, positive ionization

In [23]:
df_data = pd.read_csv(directory+filePOSdata, delimiter="\t")

df_data["mass_to_charge"] = df_data["mass_to_charge"].astype(str)
df_data["retention_time"] = df_data["retention_time"].astype(str)
# Create a new column to act as unique identifier
df_data['feature_identifier'] = df_data[['retention_time', 'mass_to_charge']].agg('_'.join, axis=1)
df_data

,database_identifier,chemical_formula,smiles,inchi,metabolite_identification,mass_to_charge,fragmentation,modifications,charge,retention_time,...,20080809_NIH_EWY_POS_120001,20080809_NIH_EWY_POS_120801,20080809_NIH_EWY_POS_120901,20080809_NIH_EWY_POS_121001,20080809_NIH_EWY_POS_121101,20080809_NIH_EWY_POS_121301,20080809_NIH_EWY_POS_121401,20080809_NIH_EWY_POS_121501,20080809_NIH_EWY_POS_121601,feature_identifier
0,NaN,NaN,NaN,NaN,NaN,61.04057539,NaN,NaN,NaN,23.69620098,...,966.287220,604.518478,986.565661,700.388908,1960.525759,1418.422841,906.491208,959.062966,927.906647,23.69620098_61.04057539
1,NaN,NaN,NaN,NaN,NaN,65.03960654,NaN,NaN,NaN,194.3244535,...,916.014650,1248.749154,11.298380,900.534151,45.460392,19.116902,275.644713,618.988805,386.077962,194.3244535_65.03960654
2,NaN,NaN,NaN,NaN,NaN,67.05509633,NaN,NaN,NaN,129.6460141,...,134.714877,135.943514,136.173812,114.255138,159.034005,206.173812,115.236635,180.022781,161.134571,129.6460141_67.05509633
3,NaN,NaN,NaN,NaN,NaN,69.03439623,NaN,NaN,NaN,134.5892937,...,288.324348,592.785685,745.227576,330.629247,526.655055,637.258304,298.995057,504.387782,334.728824,134.5892937_69.03439623
4,NaN,NaN,NaN,NaN,NaN,69.07064049,NaN,NaN,NaN,168.1367301,...,1261.136852,2121.767797,770.473328,989.594172,1460.290527,3287.041663,662.838214,1689.439386,1359.385626,168.1367301_69.07064049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1802,NaN,NaN,NaN,NaN,NaN,709.1183383,NaN,NaN,NaN,2.566976525,...,829.592352,18.506501,12.104405,24.294763,23.729870,11.429179,1114.272358,35.975881,38.736288,2.566976525_709.1183383
1803,NaN,NaN,NaN,NaN,NaN,815.3149438,NaN,NaN,NaN,163.7460356,...,521.087231,369.445745,362.603034,380.982115,597.316254,308.084919,482.706958,301.744358,342.017830,163.7460356_815.3149438
1804,NaN,NaN,NaN,NaN,NaN,816.3187089,NaN,NaN,NaN,163.7243689,...,287.085687,194.637294,172.415999,179.251419,310.023278,151.924427,274.318137,198.515852,158.653270,163.7243689_816.3187089
1805,NaN,NaN,NaN,NaN,NaN,846.2531288,NaN,NaN,NaN,163.7428656,...,301.902106,205.976048,208.244785,187.923451,317.514425,143.146336,236.891887,221.759640,191.185094,163.7428656_846.2531288


In [24]:
# Find the index where to cut to keep only samples columns
df_data.iloc[:, 21:]

,20080809_NIH_EWY_POS_000801,20080809_NIH_EWY_POS_000901,20080809_NIH_EWY_POS_001001,20080809_NIH_EWY_POS_001101,20080809_NIH_EWY_POS_001201,20080809_NIH_EWY_POS_001301,20080809_NIH_EWY_POS_001401,20080809_NIH_EWY_POS_001701,20080809_NIH_EWY_POS_001801,20080809_NIH_EWY_POS_001901,...,20080809_NIH_EWY_POS_120001,20080809_NIH_EWY_POS_120801,20080809_NIH_EWY_POS_120901,20080809_NIH_EWY_POS_121001,20080809_NIH_EWY_POS_121101,20080809_NIH_EWY_POS_121301,20080809_NIH_EWY_POS_121401,20080809_NIH_EWY_POS_121501,20080809_NIH_EWY_POS_121601,feature_identifier
0,737.287875,1741.034914,1361.404492,274.404961,1263.557468,946.303138,447.245824,589.444170,913.687036,1033.145582,...,966.287220,604.518478,986.565661,700.388908,1960.525759,1418.422841,906.491208,959.062966,927.906647,23.69620098_61.04057539
1,10.723209,18.146307,1105.049875,9.240218,19.567701,15.392333,149.940102,16.808794,20.478742,13.707881,...,916.014650,1248.749154,11.298380,900.534151,45.460392,19.116902,275.644713,618.988805,386.077962,194.3244535_65.03960654
2,195.055942,145.981269,145.404688,149.927454,69.078500,119.981894,183.997869,139.674066,100.720331,125.731920,...,134.714877,135.943514,136.173812,114.255138,159.034005,206.173812,115.236635,180.022781,161.134571,129.6460141_67.05509633
3,416.863596,385.196403,417.008741,814.330851,250.909533,314.991208,301.596626,55.519079,232.679165,339.016848,...,288.324348,592.785685,745.227576,330.629247,526.655055,637.258304,298.995057,504.387782,334.728824,134.5892937_69.03439623
4,712.816868,1202.201000,1131.424347,162.341896,530.681209,533.684313,817.056818,406.009481,831.992126,955.695018,...,1261.136852,2121.767797,770.473328,989.594172,1460.290527,3287.041663,662.838214,1689.439386,1359.385626,168.1367301_69.07064049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1802,20.959649,15.486904,15.059590,32.402898,7.521822,25.824395,29.693833,25.266455,22.400962,43.983356,...,829.592352,18.506501,12.104405,24.294763,23.729870,11.429179,1114.272358,35.975881,38.736288,2.566976525_709.1183383
1803,678.954407,755.757372,748.539285,36.671668,986.381185,422.426809,426.230351,587.061162,807.892115,853.727116,...,521.087231,369.445745,362.603034,380.982115,597.316254,308.084919,482.706958,301.744358,342.017830,163.7460356_815.3149438
1804,373.229426,382.569005,416.008507,189.943377,536.156003,253.903410,204.605861,328.758926,519.462557,425.260430,...,287.085687,194.637294,172.415999,179.251419,310.023278,151.924427,274.318137,198.515852,158.653270,163.7243689_816.3187089
1805,454.539318,712.655450,894.248360,213.509763,1077.612565,505.970115,248.268449,483.273472,676.940831,779.920743,...,301.902106,205.976048,208.244785,187.923451,317.514425,143.146336,236.891887,221.759640,191.185094,163.7428656_846.2531288


In [25]:
# Create a new df with only the samples and set the identifying column as index
df_forML = df_data[list(df_data.columns)[21:]].copy()
df_forML.set_index("feature_identifier", inplace=True)

In [26]:
df_forML

,20080809_NIH_EWY_POS_000801,20080809_NIH_EWY_POS_000901,20080809_NIH_EWY_POS_001001,20080809_NIH_EWY_POS_001101,20080809_NIH_EWY_POS_001201,20080809_NIH_EWY_POS_001301,20080809_NIH_EWY_POS_001401,20080809_NIH_EWY_POS_001701,20080809_NIH_EWY_POS_001801,20080809_NIH_EWY_POS_001901,...,20080809_NIH_EWY_POS_119901,20080809_NIH_EWY_POS_120001,20080809_NIH_EWY_POS_120801,20080809_NIH_EWY_POS_120901,20080809_NIH_EWY_POS_121001,20080809_NIH_EWY_POS_121101,20080809_NIH_EWY_POS_121301,20080809_NIH_EWY_POS_121401,20080809_NIH_EWY_POS_121501,20080809_NIH_EWY_POS_121601
feature_identifier,,,,,,,,,,,,,,,,,,,,,
23.69620098_61.04057539,737.287875,1741.034914,1361.404492,274.404961,1263.557468,946.303138,447.245824,589.444170,913.687036,1033.145582,...,794.703338,966.287220,604.518478,986.565661,700.388908,1960.525759,1418.422841,906.491208,959.062966,927.906647
194.3244535_65.03960654,10.723209,18.146307,1105.049875,9.240218,19.567701,15.392333,149.940102,16.808794,20.478742,13.707881,...,723.053630,916.014650,1248.749154,11.298380,900.534151,45.460392,19.116902,275.644713,618.988805,386.077962
129.6460141_67.05509633,195.055942,145.981269,145.404688,149.927454,69.078500,119.981894,183.997869,139.674066,100.720331,125.731920,...,60.190658,134.714877,135.943514,136.173812,114.255138,159.034005,206.173812,115.236635,180.022781,161.134571
134.5892937_69.03439623,416.863596,385.196403,417.008741,814.330851,250.909533,314.991208,301.596626,55.519079,232.679165,339.016848,...,259.345758,288.324348,592.785685,745.227576,330.629247,526.655055,637.258304,298.995057,504.387782,334.728824
168.1367301_69.07064049,712.816868,1202.201000,1131.424347,162.341896,530.681209,533.684313,817.056818,406.009481,831.992126,955.695018,...,481.249322,1261.136852,2121.767797,770.473328,989.594172,1460.290527,3287.041663,662.838214,1689.439386,1359.385626
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2.566976525_709.1183383,20.959649,15.486904,15.059590,32.402898,7.521822,25.824395,29.693833,25.266455,22.400962,43.983356,...,416.818621,829.592352,18.506501,12.104405,24.294763,23.729870,11.429179,1114.272358,35.975881,38.736288
163.7460356_815.3149438,678.954407,755.757372,748.539285,36.671668,986.381185,422.426809,426.230351,587.061162,807.892115,853.727116,...,312.353204,521.087231,369.445745,362.603034,380.982115,597.316254,308.084919,482.706958,301.744358,342.017830
163.7243689_816.3187089,373.229426,382.569005,416.008507,189.943377,536.156003,253.903410,204.605861,328.758926,519.462557,425.260430,...,176.025125,287.085687,194.637294,172.415999,179.251419,310.023278,151.924427,274.318137,198.515852,158.653270


In [27]:
# Transpose to obtain a matrix better suited for ML : samples as rows and features as columns
# Because of that transposition, the column name "feature_identifier" will stay in place, in the new transposed matrix it will seem as if
# the column of samples name is named "feature_identifier".  
df_forML = df_forML.T
df_forML

feature_identifier,23.69620098_61.04057539,194.3244535_65.03960654,129.6460141_67.05509633,134.5892937_69.03439623,168.1367301_69.07064049,46.95051744_69.07062811,14.94648176_70.01279282,28.98185787_71.04988884,105.9502218_72.08143376,167.9220779_73.06558241,...,73.89904859_678.1805151,23.76681642_689.1147144,66.74365521_693.1470018,67.33302818_694.1502612,64.69314377_699.1634376,2.566976525_709.1183383,163.7460356_815.3149438,163.7243689_816.3187089,163.7428656_846.2531288,163.706299_847.2570972
20080809_NIH_EWY_POS_000801,737.287875,10.723209,195.055942,416.863596,712.816868,505.975197,608.040844,1876.967936,473.540919,98.362921,...,16.883190,266.446294,43.710541,32.131352,18.611301,20.959649,678.954407,373.229426,454.539318,460.633061
20080809_NIH_EWY_POS_000901,1741.034914,18.146307,145.981269,385.196403,1202.201000,93.122454,597.691997,1073.218071,797.681264,202.307981,...,13.027623,302.480276,62.984525,32.638777,17.635289,15.486904,755.757372,382.569005,712.655450,702.998433
20080809_NIH_EWY_POS_001001,1361.404492,1105.049875,145.404688,417.008741,1131.424347,112.035930,690.963707,1211.852843,206.302472,220.343086,...,5.647932,69.968527,19.362430,25.171161,9.969408,15.059590,748.539285,416.008507,894.248360,881.176306
20080809_NIH_EWY_POS_001101,274.404961,9.240218,149.927454,814.330851,162.341896,104.009710,275.799318,1263.033799,1029.768335,25.150571,...,11.673831,67.101383,18.343664,29.136921,18.762967,32.402898,36.671668,189.943377,213.509763,209.312066
20080809_NIH_EWY_POS_001201,1263.557468,19.567701,69.078500,250.909533,530.681209,77.977104,471.602211,350.667552,158.053281,110.982771,...,3.012086,171.029135,4.569846,16.825130,4.494865,7.521822,986.381185,536.156003,1077.612565,903.720227
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20080809_NIH_EWY_POS_121101,1960.525759,45.460392,159.034005,526.655055,1460.290527,152.529587,856.787464,827.074836,575.590295,229.514895,...,22.901151,354.573920,32.832008,23.737287,12.514118,23.729870,597.316254,310.023278,317.514425,220.734215
20080809_NIH_EWY_POS_121301,1418.422841,19.116902,206.173812,637.258304,3287.041663,256.074750,954.765658,724.911924,1146.301994,385.996899,...,16.467069,194.592730,29.403825,28.045017,15.295619,11.429179,308.084919,151.924427,143.146336,137.952942
20080809_NIH_EWY_POS_121401,906.491208,275.644713,115.236635,298.995057,662.838214,98.407530,758.066154,522.728510,276.742198,114.596657,...,1981.821963,201.239896,8909.392648,2909.007094,1739.269839,1114.272358,482.706958,274.318137,236.891887,307.624676
20080809_NIH_EWY_POS_121501,959.062966,618.988805,180.022781,504.387782,1689.439386,95.691910,1231.064184,918.890009,500.722196,271.065077,...,19.116056,283.746931,32.063899,39.289416,12.561822,35.975881,301.744358,198.515852,221.759640,219.666255


In [28]:
# Save the df of POS samples to csv file
df_forML.to_csv(directory+"MTBLS28_data_POS_forML.csv", sep=";")

In [29]:
# Remove the date and 'POS' from the samples name, but keep the "_" so names stay as strings
# It is done to be able to match with samples from negative ionization
df_forML.index = list(map(methodcaller("replace", "20080809_NIH_EWY_POS", ""), df_forML.index))
df_forML

feature_identifier,23.69620098_61.04057539,194.3244535_65.03960654,129.6460141_67.05509633,134.5892937_69.03439623,168.1367301_69.07064049,46.95051744_69.07062811,14.94648176_70.01279282,28.98185787_71.04988884,105.9502218_72.08143376,167.9220779_73.06558241,...,73.89904859_678.1805151,23.76681642_689.1147144,66.74365521_693.1470018,67.33302818_694.1502612,64.69314377_699.1634376,2.566976525_709.1183383,163.7460356_815.3149438,163.7243689_816.3187089,163.7428656_846.2531288,163.706299_847.2570972
_000801,737.287875,10.723209,195.055942,416.863596,712.816868,505.975197,608.040844,1876.967936,473.540919,98.362921,...,16.883190,266.446294,43.710541,32.131352,18.611301,20.959649,678.954407,373.229426,454.539318,460.633061
_000901,1741.034914,18.146307,145.981269,385.196403,1202.201000,93.122454,597.691997,1073.218071,797.681264,202.307981,...,13.027623,302.480276,62.984525,32.638777,17.635289,15.486904,755.757372,382.569005,712.655450,702.998433
_001001,1361.404492,1105.049875,145.404688,417.008741,1131.424347,112.035930,690.963707,1211.852843,206.302472,220.343086,...,5.647932,69.968527,19.362430,25.171161,9.969408,15.059590,748.539285,416.008507,894.248360,881.176306
_001101,274.404961,9.240218,149.927454,814.330851,162.341896,104.009710,275.799318,1263.033799,1029.768335,25.150571,...,11.673831,67.101383,18.343664,29.136921,18.762967,32.402898,36.671668,189.943377,213.509763,209.312066
_001201,1263.557468,19.567701,69.078500,250.909533,530.681209,77.977104,471.602211,350.667552,158.053281,110.982771,...,3.012086,171.029135,4.569846,16.825130,4.494865,7.521822,986.381185,536.156003,1077.612565,903.720227
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
_121101,1960.525759,45.460392,159.034005,526.655055,1460.290527,152.529587,856.787464,827.074836,575.590295,229.514895,...,22.901151,354.573920,32.832008,23.737287,12.514118,23.729870,597.316254,310.023278,317.514425,220.734215
_121301,1418.422841,19.116902,206.173812,637.258304,3287.041663,256.074750,954.765658,724.911924,1146.301994,385.996899,...,16.467069,194.592730,29.403825,28.045017,15.295619,11.429179,308.084919,151.924427,143.146336,137.952942
_121401,906.491208,275.644713,115.236635,298.995057,662.838214,98.407530,758.066154,522.728510,276.742198,114.596657,...,1981.821963,201.239896,8909.392648,2909.007094,1739.269839,1114.272358,482.706958,274.318137,236.891887,307.624676
_121501,959.062966,618.988805,180.022781,504.387782,1689.439386,95.691910,1231.064184,918.890009,500.722196,271.065077,...,19.116056,283.746931,32.063899,39.289416,12.561822,35.975881,301.744358,198.515852,221.759640,219.666255


### Data, negative ionization

In [30]:
# Repete whole process from POS data
# Creating the feature identifier (RT_mz), keep only samples columns, transpose, save matrix in file, remove date and 'NEG' from name
df_data2 = pd.read_csv(directory+fileNEGdata, delimiter="\t")

df_data2["mass_to_charge"] = df_data2["mass_to_charge"].astype(str)
df_data2["retention_time"] = df_data2["retention_time"].astype(str)
# Create a new column
df_data2['feature_identifier'] = df_data2[['retention_time', 'mass_to_charge']].agg('_'.join, axis=1)

df_forML2 = df_data2[list(df_data2.columns)[21:]].copy()
df_forML2.set_index("feature_identifier", inplace=True)

df_forML2 = df_forML2.T
df_forML2.to_csv(directory+"MTBLS28_data_NEG_forML.csv", sep=";")

df_forML2.index = list(map(methodcaller("replace", "20081009_NIH_EWY_NEG", ""), df_forML2.index))

df_forML2

feature_identifier,71.42227731_59.01287386,18.06210069_61.98754668,85.44452264_67.01802423,8.790078676_69.03370296,0.00599967_73.02849529,307.7406504_73.02851982,23.05386329_74.0236941,171.6459655_74.02368404,8.628348983_74.02367351,164.9521402_76.02188479,...,67.02585893_654.1877117,203.7120549_656.1598044,65.13434094_675.1662462,30.26164649_675.9694341,65.24790222_676.1706026,129.4621742_681.9864247,74.88807647_816.9979748,159.1402212_829.2741513,59.0389688_830.2780636,159.4579693_835.2916339
_000801,449.595189,140.988391,257.341107,346.855774,688.206503,75.961242,1519.850017,453.023949,127.193440,4.320690,...,11.110621,447.062385,11.582430,361.407282,8.116612,1701.574847,654.709061,255.891005,0.000000,837.286193
_000901,296.208994,268.064718,92.254722,189.904701,498.017690,91.421550,1024.179086,383.954628,401.739271,110.103792,...,11.383106,96.258475,8.151252,293.209620,10.480181,1114.445595,942.435099,373.052922,116.591222,915.449710
_001001,403.446305,269.873101,34.485419,87.308633,648.940829,74.199538,147.793668,84.412654,89.930031,153.189181,...,6.783986,94.835221,10.080213,14.170735,5.040106,65.505655,693.364192,333.977982,156.129887,1339.103186
_001101,170.403298,17.980093,492.174607,195.882377,277.117103,85.616172,1022.621269,755.365381,212.773748,15.173282,...,8.586245,93.550079,13.259503,272.376362,7.549450,378.770165,41.126646,43.347316,23.052010,66.495486
_001201,101.926151,130.237477,124.354724,398.226377,216.464687,204.819886,356.680388,45.514516,138.909756,14.931891,...,5.365519,189.966241,1.378968,149.632420,2.757936,816.024215,1494.391236,503.185273,204.230173,1334.724331
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
_121101,431.339594,100.254003,18.397546,44.917081,239.798247,57.652031,1023.716380,219.748689,125.426683,95.682959,...,8.202468,90.431451,11.757620,499.875750,5.452049,112.427114,24.933092,428.134467,178.512502,111.593952
_121301,131.448210,74.475308,211.397025,573.008868,128.128863,55.946754,643.018290,125.650093,163.751954,86.939929,...,7.428486,96.978705,12.910084,351.847741,9.221490,84.906182,8.278706,167.444515,75.701745,26.638651
_121401,106.121202,98.211545,292.535243,81.344234,76.066341,239.719882,733.629646,132.925758,35.660266,51.575600,...,16997.368690,97.649426,1286.496840,162.165936,831.570491,51.721552,6.182449,273.377516,125.428804,50.735423
_121501,167.852004,89.042809,560.533832,423.706498,162.909921,56.723079,287.100539,162.626349,96.568789,88.815518,...,11.724701,28.887045,8.490759,176.466789,4.245379,47.303721,5.251043,181.505043,83.848711,31.885745


### Data, both/combined ionization

In [31]:
# Combine POS df with NEG df based on the samples name : column "feature_identifier" (previously set as index)
# Reminder : because of the transposition, the column name "feature_identifier" stays in place, and the column of samples name is named "feature_identifier". 
df_combined_forML = pd.merge(df_forML, df_forML2, right_index=True, left_index=True)
df_combined_forML

feature_identifier,23.69620098_61.04057539,194.3244535_65.03960654,129.6460141_67.05509633,134.5892937_69.03439623,168.1367301_69.07064049,46.95051744_69.07062811,14.94648176_70.01279282,28.98185787_71.04988884,105.9502218_72.08143376,167.9220779_73.06558241,...,67.02585893_654.1877117,203.7120549_656.1598044,65.13434094_675.1662462,30.26164649_675.9694341,65.24790222_676.1706026,129.4621742_681.9864247,74.88807647_816.9979748,159.1402212_829.2741513,59.0389688_830.2780636,159.4579693_835.2916339
_000801,737.287875,10.723209,195.055942,416.863596,712.816868,505.975197,608.040844,1876.967936,473.540919,98.362921,...,11.110621,447.062385,11.582430,361.407282,8.116612,1701.574847,654.709061,255.891005,0.000000,837.286193
_000901,1741.034914,18.146307,145.981269,385.196403,1202.201000,93.122454,597.691997,1073.218071,797.681264,202.307981,...,11.383106,96.258475,8.151252,293.209620,10.480181,1114.445595,942.435099,373.052922,116.591222,915.449710
_001001,1361.404492,1105.049875,145.404688,417.008741,1131.424347,112.035930,690.963707,1211.852843,206.302472,220.343086,...,6.783986,94.835221,10.080213,14.170735,5.040106,65.505655,693.364192,333.977982,156.129887,1339.103186
_001101,274.404961,9.240218,149.927454,814.330851,162.341896,104.009710,275.799318,1263.033799,1029.768335,25.150571,...,8.586245,93.550079,13.259503,272.376362,7.549450,378.770165,41.126646,43.347316,23.052010,66.495486
_001201,1263.557468,19.567701,69.078500,250.909533,530.681209,77.977104,471.602211,350.667552,158.053281,110.982771,...,5.365519,189.966241,1.378968,149.632420,2.757936,816.024215,1494.391236,503.185273,204.230173,1334.724331
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
_121101,1960.525759,45.460392,159.034005,526.655055,1460.290527,152.529587,856.787464,827.074836,575.590295,229.514895,...,8.202468,90.431451,11.757620,499.875750,5.452049,112.427114,24.933092,428.134467,178.512502,111.593952
_121301,1418.422841,19.116902,206.173812,637.258304,3287.041663,256.074750,954.765658,724.911924,1146.301994,385.996899,...,7.428486,96.978705,12.910084,351.847741,9.221490,84.906182,8.278706,167.444515,75.701745,26.638651
_121401,906.491208,275.644713,115.236635,298.995057,662.838214,98.407530,758.066154,522.728510,276.742198,114.596657,...,16997.368690,97.649426,1286.496840,162.165936,831.570491,51.721552,6.182449,273.377516,125.428804,50.735423
_121501,959.062966,618.988805,180.022781,504.387782,1689.439386,95.691910,1231.064184,918.890009,500.722196,271.065077,...,11.724701,28.887045,8.490759,176.466789,4.245379,47.303721,5.251043,181.505043,83.848711,31.885745


In [32]:
# Save the df of combined POS and NEG data to csv file
df_combined_forML.to_csv(directory+"MTBLS28_CombinedData2_forML.csv", sep=";")

### Metadata, both/combined ionization

In [33]:
df_metadata = df_metadata.rename(columns={'Sample Name': 'Sample_Name'})
df_metadata = df_metadata[~df_metadata.Sample_Name.str.contains("NEG")]  # Remove lines based on the Sample_Name column content (remove if NEG is present)
df_metadata

,Source Name,Characteristics[Organism],Term Source REF,Term Accession Number,Characteristics[Organism part],Term Source REF.1,Term Accession Number.1,Characteristics[Variant],Term Source REF.2,Term Accession Number.2,...,Term Accession Number.4,Factor Value[Race],Term Source REF.5,Term Accession Number.5,Factor Value[Gender],Term Source REF.6,Term Accession Number.6,Factor Value[Sample Type],Term Source REF.7,Term Accession Number.7
0,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Male,NaN,NaN,Control,NaN,NaN
1,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Female,NaN,NaN,Case,NaN,NaN
2,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Male,NaN,NaN,Control,NaN,NaN
3,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Male,NaN,NaN,Control,NaN,NaN
4,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Female,NaN,NaN,Control,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1000,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,African American,NaN,NaN,Female,NaN,NaN,Case,NaN,NaN
1001,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Male,NaN,NaN,Control,NaN,NaN
1002,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Male,NaN,NaN,Case,NaN,NaN
1003,urine,Homo sapiens,NCBITAXON,http://purl.bioontology.org/ontology/NCBITAXON...,urine,BTO,http://purl.obolibrary.org/obo/BTO_0001419,NaN,NaN,NaN,...,NaN,European American,NaN,NaN,Female,NaN,NaN,Control,NaN,NaN


In [34]:
df_metadata["Sample_Name"] = df_metadata["Sample_Name"].str.replace("20080809_NIH_EWY_POS","")

In [35]:
df_metadata.iloc[:, 10:20]

,Characteristics[Sample type],Term Source REF.3,Term Accession Number.3,Protocol REF,Sample_Name,Factor Value[Smoking],Term Source REF.4,Term Accession Number.4,Factor Value[Race],Term Source REF.5
0,NaN,NaN,NaN,Sample collection,_000801,Current Smoker,NaN,NaN,European American,NaN
1,NaN,NaN,NaN,Sample collection,_000901,Former Smoker,NaN,NaN,European American,NaN
2,NaN,NaN,NaN,Sample collection,_001001,Former Smoker,NaN,NaN,European American,NaN
3,NaN,NaN,NaN,Sample collection,_001101,Never Smoker,NaN,NaN,European American,NaN
4,NaN,NaN,NaN,Sample collection,_001201,Never Smoker,NaN,NaN,European American,NaN
...,...,...,...,...,...,...,...,...,...,...
1000,NaN,NaN,NaN,Sample collection,_121101,Current Smoker,NaN,NaN,African American,NaN
1001,NaN,NaN,NaN,Sample collection,_121301,Former Smoker,NaN,NaN,European American,NaN
1002,NaN,NaN,NaN,Sample collection,_121401,Current Smoker,NaN,NaN,European American,NaN
1003,NaN,NaN,NaN,Sample collection,_121501,Former Smoker,NaN,NaN,European American,NaN


In [36]:
# Save df of metadata from both ionization to csv file
df_metadata.to_csv(directory+"s_MTBLS28_merged.csv", sep=";")